In [1]:
import polars as pl
import pandas as pd
import numpy as np

In [2]:
data = pl.read_parquet("/sdd/Dubaoset/src/Phong/Model/data/trainDistributed/Choosen/total_5_clean.parquet").to_pandas()

In [3]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.2, random_state=42)
del data

In [4]:
class0 = np.sum(test["output_5"] == 0)
class1 = np.sum(test["output_5"] == 1)

In [5]:
class0

np.int64(858218)

In [6]:
class1

np.int64(6282)

In [ ]:
outputLabel = ["output_0", "output_1", "output_2", "output_3", "output_4", "output_5"]

In [6]:
class0 = 0
class1 = 0

In [7]:
for out in outputLabel:
    class0 += np.sum(data[out] == 0)
    class1 += np.sum(data[out] == 1)

In [8]:
class0

np.int64(11348855)

In [9]:
class1

np.int64(800779)

In [3]:
fullCol = list(data.columns)

In [4]:
# Check liệu rằng có cột nào bị giá trị nan hay không và in tên cột đó ra
for col in fullCol:
    if data[col].isna().sum() > 0:
        print(f"Cột {col} có {data[col].isna().sum()} giá trị nan")

Cột B04B_t-6 có 28784 giá trị nan
Cột B05B_t-6 có 28784 giá trị nan
Cột B06B_t-6 có 28784 giá trị nan
Cột B09B_t-6 có 28784 giá trị nan
Cột B10B_t-6 có 28784 giá trị nan
Cột B11B_t-6 có 28784 giá trị nan
Cột B12B_t-6 có 28784 giá trị nan
Cột B14B_t-6 có 28784 giá trị nan
Cột B16B_t-6 có 28784 giá trị nan
Cột I2B_t-6 có 28784 giá trị nan
Cột I4B_t-6 có 28784 giá trị nan
Cột IRB_t-6 có 28784 giá trị nan
Cột VSB_t-6 có 28784 giá trị nan
Cột WVB_t-6 có 28784 giá trị nan
Cột B04B_t-5 có 28858 giá trị nan
Cột B05B_t-5 có 28858 giá trị nan
Cột B06B_t-5 có 28858 giá trị nan
Cột B09B_t-5 có 28858 giá trị nan
Cột B10B_t-5 có 28858 giá trị nan
Cột B11B_t-5 có 28858 giá trị nan
Cột B12B_t-5 có 28858 giá trị nan
Cột B14B_t-5 có 28858 giá trị nan
Cột B16B_t-5 có 28858 giá trị nan
Cột I2B_t-5 có 28858 giá trị nan
Cột I4B_t-5 có 28858 giá trị nan
Cột IRB_t-5 có 28858 giá trị nan
Cột VSB_t-5 có 28858 giá trị nan
Cột WVB_t-5 có 28858 giá trị nan
Cột B04B_t-4 có 29197 giá trị nan
Cột B05B_t-4 có 29197 gi

In [5]:
import os
import torch
import wandb
import sys
sys.path.append("/sdd/Dubaoset/src/Phong/Model/TrainingCode")
from Model import LSTM
from FocalLoss import FocalLoss

class inputParameters:
    def __init__(self):
        # "/sdd/Dubaoset/src/Phong/Model/data/trainDistributed/Choosen"
        # "/sdd/Dubaoset/src/Phong/Model/data/validation"
        self.inputFolder = "/sdd/Dubaoset/src/Phong/Model/data/trainDistributed/Choosen"
        self.inputVal = "/sdd/Dubaoset/src/Phong/Model/data/validation"
        self.inputInfo = "/sdd/Dubaoset/src/Phong/Model/TrainingCode/infoJsonl/total_5_scales.jsonl"

        self.choosenFile = [
            os.path.join(self.inputFolder, item) for item in os.listdir(self.inputFolder) if item.endswith('.parquet')
        ]

        self.choosenVal = [
            os.path.join(self.inputVal, item) for item in os.listdir(self.inputVal) if item.endswith('.parquet')
        ]
        self.fullBand = ['B09B','B10B','B11B','B12B','B14B','B16B','I2B','I4B','IRB','WVB', 'NDVI', 'Dem_value', 'NDVIIsLand', 'DEMIsLand']
        self.diffBand = None
        self.exceptBand = ['Dem_value', 'DEMIsLand']

        self.outputLabel = ["output_0", "output_1", "output_2", "output_3", "output_4", "output_5"]
        self.timeStamps = 6
        self.outputMetrics = "/sdd/Dubaoset/src/Phong/Model/data/trainDistributed/logFolder/mayMetrics.log"
        self.modelName = "mayTrain.pth"

        self.device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
        print(self.device)
        self.epochs = 150
        self.lr = 0.0001
        self.weight_decay = 0.0001
        self.batch_size = 128
        self.threshold = 0.5
        self.hidden_size= 256
        self.dropout= 0.2
        self.num_layer= 2
        self.output_features= 1
        self.alpha = 0.5
        self.gamma = 2
        self.patience = 20

        # wandb.login(key="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")
        # self.run = wandb.init(
        #     project= "trainNewDistribution",
        #     # Track hyperparameters and run metadata.
        #     config = {
        #         "epochs": self.epochs,
        #         "batch_size": self.batch_size,
        #         "lr": self.lr,
        #         "dropout": self.dropout,
        #         "weight_decay": self.weight_decay,
        #         "hidden_size": self.hidden_size,
        #         "num_layer": self.num_layer,
        #         "input_size": len(self.fullBand),
        #         "output_features": self.output_features,
        #     }
        # )
        torch.manual_seed(42)
        self.myModel = LSTM(input_size= len(self.fullBand), hidden_size= self.hidden_size, dropout= self.dropout, num_layer= self.num_layer, output_features= self.output_features).to(self.device)
        self.loss_fn = FocalLoss(alpha= self.alpha, gamma= self.gamma)
        self.optimizer = torch.optim.AdamW(self.myModel.parameters(),lr= self.lr, weight_decay= self.weight_decay)
        self.scheduler = None
        self.model_dir = os.path.join("/sdd/Dubaoset/src/Phong/Model/LSTM6Output_Way1", self.modelName)


In [6]:
import torch
import sys
sys.path.append("/sdd/Dubaoset/src/Phong/Model/TrainingCode")
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,TensorDataset
import logging
import pandas as pd
import numpy as np
from configData import createDataForAnalyst
import polars as pl
from tqdm import tqdm
import json

def returnTestDataset(testDataFrame, device, batch_size, timestamps, outputLabel, exceptBand, fullBand):
    bandType = [
        f"{band}_t{i:+d}" if band not in exceptBand else band
        for i in range(-timestamps, timestamps)
        for band in fullBand
    ]

    XTestInput, yTestLabel = testDataFrame.loc[:,bandType].values.astype('float32'), testDataFrame.loc[:,outputLabel].values.astype('float32')
    del testDataFrame

    X_tensor = torch.tensor(XTestInput.reshape(-1, timestamps * 2, len(bandType) // (timestamps * 2)), dtype= torch.float32, device= device)
    y_tensor = torch.tensor(yTestLabel, dtype= torch.float32, device= device)

    del XTestInput, yTestLabel
    TensorDSTest = TensorDataset(X_tensor, y_tensor)

    del X_tensor, y_tensor
    testDataset = DataLoader(
        TensorDSTest,
        batch_size=batch_size,
        shuffle=False
    )

    return testDataset

# Tạo band diff, normalize band, tạo dataset cho tập train và tập test
def loadedFullDataset(inputFileList, diffBand, exceptBand, timeStamps, inputInfo, fullBand):
    dataFrames = []
    for file in tqdm(inputFileList, total= len(inputFileList)):
        df = pl.read_parquet(file).to_pandas()
        dataFrames.append(df)
    
    # Load vào để lấy các key có thể được dùng để chuẩn hóa.
    listOfBandInfo = {}
    with open(inputInfo, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            listOfBandInfo.update(data)
    
    fullDataSet = pd.concat(dataFrames, ignore_index= True)
    del dataFrames
    dataObject = createDataForAnalyst(fullDataSet, timeStamps, inputInfo)
    if diffBand is not None:
        dataObject.createDiffBand(diffBand)
    for band in fullBand:
        # Hàm lấy min max value
        if band in listOfBandInfo:
            if band not in exceptBand:
                dataObject.normalBand(band, exceptBand, "MinMaxScaler", keyValue= ["min", "max"])
            else:
                dataObject.normalBand(band, exceptBand, "MaxScaler", keyValue= ["max"])
    print("Create diff band completed")
    return dataObject.inputDf

# Tạo dataset 6 thời gian cho tập train.
def handleTrainData(trainDataset, fullBand, exceptBand, timeStamps, outputLabel):
    features = [
        f"{band}_t{i:+d}" if band not in exceptBand else band
        for i in range(-timeStamps, 0)
        for band in fullBand
    ]

    features_unique = list(dict.fromkeys(features))
    outputs = [outputLabel[0]]  # Chỉ lấy label ở thời điểm t0
    newCols = features_unique + outputs

    chunks = []  # Giữ list of DataFrames thay vì list of numpy rows

    for i in range(timeStamps):
        period = [
            f"{band}_t{j:+d}" if band not in exceptBand else band
            for j in range(i - timeStamps, i)
            for band in fullBand
        ]

        period_unique = list(dict.fromkeys(period))
        output = [outputLabel[i]]

        # Chỉ lấy đúng cột cần, rename để khớp newCols
        chunk = trainDataset[period_unique + output].copy()
        chunk.columns = newCols
        chunks.append(chunk)

        # Drop in-place để không giữ bản copy
        columns = []
        for band in fullBand:
            if band not in exceptBand:
                columns.append(f"{band}_t{i - timeStamps}")
            else:
                continue
        columns += output

        trainDataset.drop(
            columns= columns,
            inplace=True,
        )
    del trainDataset  # Giải phóng ngay sau khi xử lý xong
    fullResult = pd.concat(chunks).sample(frac=1, random_state=42).reset_index(drop=True)
    del chunks  # Giải phóng ngay sau khi concat xong

    print("Config train data completed")
    return features, outputs, fullResult

def returnTrainValDataset(trainValDataset, exceptBand, device, batch_size, timestamps, outputLabel, fullBand):
    # Train test split
    # Không shuffle để đảm bảo mỗi chunk khi lặp lại không bị trộn dữ liệu train và test
    # Chỉ shuffle batch train
    bandType = [
        f"{band}_t{i:+d}" if band not in exceptBand else band
        for i in range(-timestamps, timestamps)
        for band in fullBand
    ]
    trainDataset, valDataset = train_test_split(trainValDataset ,test_size= 0.2, random_state= 42)

    # Choose the columns that we want it to be features and labels
    features, outputs, trainDataset = handleTrainData(trainDataset, fullBand, exceptBand, timestamps, outputLabel)
    XTrainInput, yTrainLabel = trainDataset.loc[:,features], trainDataset.loc[:, outputs[0]] # features and label first
    XValInput, yValLabel = valDataset.loc[:,bandType], valDataset.loc[:,outputLabel] #  features and label
    del trainDataset, valDataset

    # We change input format into (Batch, num of units, features) which has datatype is tensor
    # Split to format(num of units,features) first
    # numpy -> format(num of units,features) -> tensor
    XTrainInput = XTrainInput.values.astype('float32')
    XValInput = XValInput.values.astype('float32')

    yTrainLabel = yTrainLabel.values.astype('float32')
    yValLabel = yValLabel.values.astype('float32')

    yTrainLabel = torch.tensor(yTrainLabel, dtype=torch.float32, device=device)
    yValLabel = torch.tensor(yValLabel, dtype=torch.float32, device=device)

    # 6 khoảng thời gian, 10 bands -> 33 bands
    XTrainInput = torch.tensor(XTrainInput.reshape(-1, timestamps, len(bandType) // (timestamps * 2)), dtype=torch.float32, device=device)
    # 12 khoảng thời gian, 10 bands -> 33 bands
    XValInput = torch.tensor(XValInput.reshape(-1, timestamps * 2, len(bandType) // (timestamps * 2)), dtype=torch.float32, device=device)

    # Combine to make a tensor-dataset and split it into small batch
    TensorDSTrain = TensorDataset(XTrainInput, yTrainLabel)
    TensorDSVal = TensorDataset(XValInput, yValLabel)

    del XTrainInput, XValInput, yTrainLabel, yValLabel

    train_dataset = DataLoader(
        TensorDSTrain,
        batch_size= batch_size,
        shuffle=True
    )

    val_dataset = DataLoader(
        TensorDSVal,
        batch_size= batch_size,
        shuffle=False
    )

    del TensorDSTrain, TensorDSVal
    return train_dataset, val_dataset

def returnDataset(trainDataset, exceptBand, device, batch_size, timestamps, outputLabel, fullBand):
    bandType = [
        f"{band}_t{i:+d}" if band not in exceptBand else band
        for i in range(-timestamps, timestamps)
        for band in fullBand
    ]
    # Choose the columns that we want it to be features and labels
    features, outputs, trainDataset = handleTrainData(trainDataset, fullBand, exceptBand, timestamps, outputLabel)
    XTrainInput, yTrainLabel = trainDataset.loc[:,features], trainDataset.loc[:, outputs[0]] # features and label first
    print(len(list(XTrainInput.columns)))
    print(XTrainInput.columns)
    del trainDataset

    # We change input format into (Batch, num of units, features) which has datatype is tensor
    # Split to format(num of units,features) first
    # numpy -> format(num of units,features) -> tensor
    XTrainInput = XTrainInput.values.astype('float32')
    yTrainLabel = yTrainLabel.values.astype('float32')

    yTrainLabel = torch.tensor(yTrainLabel, dtype=torch.float32, device=device)
    # 6 khoảng thời gian, 10 bands -> 33 bands
    XTrainInput = torch.tensor(XTrainInput.reshape(-1, timestamps, len(bandType) // (timestamps * 2)), dtype=torch.float32, device=device)

    # Combine to make a tensor-dataset and split it into small batch
    TensorDSTrain = TensorDataset(XTrainInput, yTrainLabel)
    del XTrainInput, yTrainLabel

    train_dataset = DataLoader(
        TensorDSTrain,
        batch_size= batch_size,
        shuffle=True
    )

    del TensorDSTrain
    return train_dataset

In [7]:
import sys
sys.path.append("/sdd/Dubaoset/src/Phong/Model/TrainingCode")
from TrainLoop import trainLoop

params = inputParameters()
    
print("Input parameters initialized successfully.")

trainDataset = loadedFullDataset(inputFileList= params.choosenFile, diffBand= params.diffBand, exceptBand= params.exceptBand, timeStamps= params.timeStamps, inputInfo= params.inputInfo, fullBand= params.fullBand)
valDataset = loadedFullDataset(inputFileList= params.choosenVal, diffBand= params.diffBand, exceptBand= params.exceptBand, timeStamps= params.timeStamps, inputInfo= params.inputInfo, fullBand= params.fullBand)
print("Datasets loaded successfully.")

train_dataset = returnDataset(
    trainDataset= trainDataset, exceptBand= params.exceptBand, device= params.device, batch_size= params.batch_size, timestamps= params.timeStamps, outputLabel= params.outputLabel, fullBand= params.fullBand
)

val_dataset = returnTestDataset(
    testDataFrame= valDataset, device= params.device, batch_size= params.batch_size, timestamps= params.timeStamps, outputLabel= params.outputLabel, exceptBand= params.exceptBand, fullBand= params.fullBand
)

print("Create train and validation dataloader successfully.")

cpu
Input parameters initialized successfully.


100%|██████████| 1/1 [01:02<00:00, 62.70s/it]


Create diff band completed


100%|██████████| 1/1 [01:40<00:00, 100.43s/it]


Create diff band completed
Datasets loaded successfully.


KeyboardInterrupt: 

In [ ]:
# Kiểm tra liệu rằng dataloader có chứa dữ liệu NaN hay không
for batch, (X,y) in tqdm(enumerate(train_dataset), desc= "Train dataset", leave=False):
    if torch.isnan(X).any() or torch.isinf(X).any():
        print(f"Batch {batch} contains NaN or Inf values.")
        break

tensor([[[ 0.9078,  0.7362,  0.8786,  ...,  0.3807,  1.0000,  1.0000],
         [ 0.8916,  0.6726,  0.8403,  ...,  0.3807,  1.0000,  1.0000],
         [ 0.8941,  0.6756,  0.8434,  ...,  0.3807,  1.0000,  1.0000],
         [ 0.8818,  0.6240,  0.8094,  ...,  0.3807,  1.0000,  1.0000],
         [ 0.8892,  0.6582,  0.8269,  ...,  0.3807,  1.0000,  1.0000],
         [ 0.8975,  0.6966,  0.8513,  ...,  0.3807,  1.0000,  1.0000]],

        [[ 0.9376,  0.8517,  0.9183,  ...,  0.0442,  1.0000,  1.0000],
         [ 0.9341,  0.8461,  0.9215,  ...,  0.0442,  1.0000,  1.0000],
         [ 0.9341,  0.8442,  0.9218,  ...,  0.0442,  1.0000,  1.0000],
         [ 0.9346,  0.8449,  0.9232,  ...,  0.0442,  1.0000,  1.0000],
         [ 0.9341,  0.8442,  0.9199,  ...,  0.0442,  1.0000,  1.0000],
         [ 0.9332,  0.8442,  0.9199,  ...,  0.0442,  1.0000,  1.0000]],

        [[ 0.9270,  0.8128,  0.9259,  ...,  0.0000,  0.0000,  1.0000],
         [ 0.9270,  0.8128,  0.9262,  ...,  0.0000,  0.0000,  1.0000],
  